## Регрессия для IC50

In [1]:
import numpy as np
import pandas as pd

import sys
sys.path.append('../src')
from preprocessing import DatasetPreprocessor
from metrics_calculator import MetricsCalculator

from sklearn.pipeline import Pipeline
from sklearn.model_selection import train_test_split, GridSearchCV

from sklearn.preprocessing import StandardScaler
from sklearn.compose import TransformedTargetRegressor
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor


In [2]:
df = pd.read_excel('../data/raw/classicMLCourseWorkData.xlsx')
df.head()

,Unnamed: 0,"IC50, mM","CC50, mM",SI,MaxAbsEStateIndex,MaxEStateIndex,MinAbsEStateIndex,MinEStateIndex,qed,SPS,...,fr_sulfide,fr_sulfonamd,fr_sulfone,fr_term_acetylene,fr_tetrazole,fr_thiazole,fr_thiocyan,fr_thiophene,fr_unbrch_alkane,fr_urea
0,0,6.239374,175.482382,28.125000,5.094096,5.094096,0.387225,0.387225,0.417362,42.928571,...,0,0,0,0,0,0,0,0,3,0
1,1,0.771831,5.402819,7.000000,3.961417,3.961417,0.533868,0.533868,0.462473,45.214286,...,0,0,0,0,0,0,0,0,3,0
2,2,223.808778,161.142320,0.720000,2.627117,2.627117,0.543231,0.543231,0.260923,42.187500,...,0,0,0,0,0,0,0,0,3,0
3,3,1.705624,107.855654,63.235294,5.097360,5.097360,0.390603,0.390603,0.377846,41.862069,...,0,0,0,0,0,0,0,0,4,0
4,4,107.131532,139.270991,1.300000,5.150510,5.150510,0.270476,0.270476,0.429038,36.514286,...,0,0,0,0,0,0,0,0,0,0


In [3]:
# удаляем строки с пропусками
clean_df = df.dropna()
# Класс расчета метрик
metrics_helper = MetricsCalculator()

In [4]:
y = clean_df['IC50, mM']
X = clean_df.drop(columns=['IC50, mM'])

In [5]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42
)

#### Линейная регрессия

In [6]:
# базовый pipeline для регрессии
baseline_pipeline = Pipeline([
    ('preprocessor', DatasetPreprocessor(
        target_column='IC50, mM', # для корректного удаления leakage-признаков
        drop_unnamed=True, # удаление служебного столбца 'Unnamed: 0'
        drop_leakage=True,  # удаление признаков, создающих утечку данных
        drop_constant_features=True, # удаление константных признаков
        drop_corr=True, # удаление сильно коррелированных признаков
        corr_threshold=0.9, # порог корреляции для удаления
        log_features=True, # логарифмирование признаков с сильной правой асимметрией
        log_skew_threshold=2.0, # порог асимметрии (skew)
        log_min_unique_values=10 # логарифмируем только признаки с достаточным числом уникальных значений
    )),
    ('scaler', StandardScaler()),
    ('model', LinearRegression())
])

In [7]:
# обучаем pipeline на train-выборке
baseline_pipeline.fit(X_train, y_train)

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators ` for more details.","[('preprocessor', ...), ('scaler', ...), ...]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing `.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
,target_column,"'IC50, mM'"
,drop_unnamed,True
,drop_leakage,True
,drop_constant_features,True
,drop_corr,True
,corr_threshold,0.9
,log_features,True


In [8]:
# получаем предсказания модели
y_train_pred = baseline_pipeline.predict(X_train)
y_test_pred = baseline_pipeline.predict(X_test)

In [9]:
# метрики на train
train_metrics = metrics_helper.regression_metrics(y_train, y_train_pred)
# метрики на test
test_metrics = metrics_helper.regression_metrics(y_test, y_test_pred)
print('Метрики на train:')
display(train_metrics)
print('Метрики на test:')
display(test_metrics)

Метрики на train:


{'MAE': 172.47454084852006,
 'MSE': 61604.32013957908,
 'RMSE': np.float64(248.20217593643108),
 'R2': 0.5274917181651799}

Метрики на test:


{'MAE': 4122.964982010893,
 'MSE': 3030472702.216258,
 'RMSE': np.float64(55049.72935643061),
 'R2': -10970.040292450032}

Линейная регрессия с базовым pipeline показала плохой результат. На обучающей выборке модель ещё что-то объясняет, но на тестовой полностью разваливается ($R^2$ сильно отрицательный, ошибки высокие).
Вероятно, проблема в том, что таргет сильно скошен и имеет большие значения.
**Необходимо попробовать логарифмирование таргета**

In [10]:
# pipeline с логарифмированием таргета
log_pipeline = Pipeline([
    (
        'preprocessor',
        DatasetPreprocessor(
            target_column='IC50, mM',
            drop_unnamed=True,
            drop_leakage=True,
            drop_constant_features=True,
            drop_corr=True,
            corr_threshold=0.9,
            log_features=True,
            log_skew_threshold=2.0,
            log_min_unique_values=10
        )
    ),
    ('scaler', StandardScaler()),
    (
        'model',
        TransformedTargetRegressor(
            regressor=LinearRegression(),
            func=np.log1p,      # логарифмируем таргет при обучении
            inverse_func=np.expm1  # возвращаем обратно при предсказании
        )
    )
])

In [11]:
# обучаем pipeline на train-выборке
log_pipeline.fit(X_train, y_train)

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators ` for more details.","[('preprocessor', ...), ('scaler', ...), ...]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing `.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
,target_column,"'IC50, mM'"
,drop_unnamed,True
,drop_leakage,True
,drop_constant_features,True
,drop_corr,True
,corr_threshold,0.9
,log_features,True


In [12]:
# получаем предсказания модели
log_y_train_pred = log_pipeline.predict(X_train)
log_y_test_pred = log_pipeline.predict(X_test)

In [13]:
# метрики на train
log_train_metrics = metrics_helper.regression_metrics(y_train, log_y_train_pred)
# метрики на test
log_test_metrics = metrics_helper.regression_metrics(y_test, log_y_test_pred)
print('Метрики на train:')
display(log_train_metrics)
print('Метрики на test:')
display(log_test_metrics)

Метрики на train:


{'MAE': 154.63856952431323,
 'MSE': 182052.58205933054,
 'RMSE': np.float64(426.6762028275429),
 'R2': -0.3963526025049098}

Метрики на test:


{'MAE': 598.653747610069,
 'MSE': 16736907.181336794,
 'RMSE': np.float64(4091.0765308579594),
 'R2': -59.59163077864242}

Логарифмирование таргета улучшило ситуацию: ошибки заметно уменьшились, и модель стала стабильнее.

Тем не менее качество остаётся низким - даже на обучающей выборке модель не объясняет зависимость ($R^2$ отрицательный). Это значит, что проблема не только в распределении таргета, а в том, что линейная модель в целом плохо подходит для этих данных.

Необходимо попробовать более сложные модели

#### Случайный лес

In [14]:
rf_pipeline = Pipeline([
    (
        'preprocessor',
        DatasetPreprocessor(
            target_column='IC50, mM',  # текущий таргет → для удаления leakage (SI)
            drop_unnamed=True,         # удаляем служебный столбец
            drop_leakage=True,         # убираем SI
            drop_constant_features=True,  # убираем константные признаки
            drop_corr=True,            # убираем сильно коррелированные признаки
            corr_threshold=0.9,        # порог корреляции
            log_features=True,         # логарифмируем сильно скошенные признаки
            log_skew_threshold=2.0,    # порог асимметрии
            log_min_unique_values=10   # исключаем дискретные признаки
        )
    ),
    # scaler можно оставить для единообразия, но для деревьев он не обязателен
    ('scaler', StandardScaler()),
    (
        'model',
        RandomForestRegressor(
            n_estimators=100,   # количество деревьев
            max_depth=None,     # глубина деревьев (без ограничения)
            random_state=42,
            n_jobs=-1           # использовать все ядра
        )
    )
])

In [15]:
# обучаем
rf_pipeline.fit(X_train, y_train)

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators ` for more details.","[('preprocessor', ...), ('scaler', ...), ...]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing `.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
,target_column,"'IC50, mM'"
,drop_unnamed,True
,drop_leakage,True
,drop_constant_features,True
,drop_corr,True
,corr_threshold,0.9
,log_features,True


In [16]:
rf_y_train_pred = rf_pipeline.predict(X_train)
rf_y_test_pred = rf_pipeline.predict(X_test)

In [17]:
rf_train_metrics = metrics_helper.regression_metrics(y_train, rf_y_train_pred)
rf_test_metrics = metrics_helper.regression_metrics(y_test, rf_y_test_pred)

print('Метрики на train:')
display(rf_train_metrics)

print('Метрики на test:')
display(rf_test_metrics)

Метрики на train:


{'MAE': 57.54775394092627,
 'MSE': 15266.304300658874,
 'RMSE': np.float64(123.55688690096912),
 'R2': 0.8829066663063878}

Метрики на test:


{'MAE': 176.67464710648798,
 'MSE': 132146.02473759398,
 'RMSE': np.float64(363.518946875667),
 'R2': 0.5215995970453773}

RandomForest показывает значительно лучшее качество по сравнению с линейными моделями. При этом наблюдается некоторое переобучение, так как качество на обучающей выборке заметно выше, чем на тестовой.
**Возможно, результат получится улучшить путём подбора гиперпараметров**

In [18]:
# сетка гиперпараметров
param_grid = {
    'model__n_estimators': [100, 200],
    'model__max_depth': [5, 10, None],
    'model__min_samples_split': [2, 5],
    'model__min_samples_leaf': [1, 2]
}

# GridSearch с кросс-валидацией
rf_grid = GridSearchCV(
    estimator=rf_pipeline,
    param_grid=param_grid,
    cv=5,
    scoring='r2',
    n_jobs=-1,
    verbose=1
)

In [19]:
rf_grid.fit(X_train, y_train)

Fitting 5 folds for each of 24 candidates, totalling 120 fits


,"estimator estimator: estimator objectThis is assumed to implement the scikit-learn estimator interface.Either estimator needs to provide a ``score`` function,or ``scoring`` must be passed.",Pipeline(step...m_state=42))])
,"param_grid param_grid: dict or list of dictionariesDictionary with parameters names (`str`) as keys and lists ofparameter settings to try as values, or a list of suchdictionaries, in which case the grids spanned by each dictionaryin the list are explored. This enables searching over any sequenceof parameter settings.","{'model__max_depth': [5, 10, ...], 'model__min_samples_leaf': [1, 2], 'model__min_samples_split': [2, 5], 'model__n_estimators': [100, 200]}"
,"scoring scoring: str, callable, list, tuple or dict, default=NoneStrategy to evaluate the performance of the cross-validated model onthe test set.If `scoring` represents a single score, one can use:- a single string (see :ref:`scoring_string_names`);- a callable (see :ref:`scoring_callable`) that returns a single value;- `None`, the `estimator`'s :ref:`default evaluation criterion ` is used.If `scoring` represents multiple scores, one can use:- a list or tuple of unique strings;- a callable returning a dictionary where the keys are the metric names and the values are the metric scores;- a dictionary with metric names as keys and callables as values.See :ref:`multimetric_grid_search` for an example.",'r2'
,"n_jobs n_jobs: int, default=NoneNumber of jobs to run in parallel.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary `for more details... versionchanged:: v0.20 `n_jobs` default changed from 1 to None",-1
,"refit refit: bool, str, or callable, default=TrueRefit an estimator using the best found parameters on the wholedataset.For multiple metric evaluation, this needs to be a `str` denoting thescorer that would be used to find the best parameters for refittingthe estimator at the end.Where there are considerations other than maximum score inchoosing a best estimator, ``refit`` can be set to a function whichreturns the selected ``best_index_`` given ``cv_results_``. In thatcase, the ``best_estimator_`` and ``best_params_`` will be setaccording to the returned ``best_index_`` while the ``best_score_``attribute will not be available.The refitted estimator is made available at the ``best_estimator_``attribute and permits using ``predict`` directly on this``GridSearchCV`` instance.Also for multiple metric evaluation, the attributes ``best_index_``,``best_score_`` and ``best_params_`` will only be available if``refit`` is set and all of them will be determined w.r.t this specificscorer.See ``scoring`` parameter to know more about multiple metricevaluation.See :ref:`sphx_glr_auto_examples_model_selection_plot_grid_search_digits.py`to see how to design a custom selection strategy using a callablevia `refit`.See :ref:`this example`for an example of how to use ``refit=callable`` to balance modelcomplexity and cross-validated score... versionchanged:: 0.20 Support for callable added.",True
,"cv cv: int, cross-validation generator or an iterable, default=NoneDetermines the cross-validation splitting strategy.Possible inputs for cv are:- None, to use the default 5-fold cross validation,- integer, to specify the number of folds in a `(Stratified)KFold`,- :term:`CV splitter`,- An iterable yielding (train, test) splits as arrays of indices.For integer/None inputs, if the estimator is a classifier and ``y`` iseither binary or multiclass, :class:`StratifiedKFold` is used. In allother cases, :class:`KFold` is used. These splitters are instantiatedwith `shuffle=False` so the splits will be the same across calls.Refer :ref:`User Guide ` for the variouscross-validation strategies that can be used here... versionchanged:: 0.22 ``cv`` default value if None changed from 3-fold to 5-fold.",5
,"verbose verbose: intControls the verbosity: the higher, the more messages.- >1 : the computation time for each f

In [20]:
# лучшие параметры
rf_grid.best_params_

{'model__max_depth': 10,
 'model__min_samples_leaf': 2,
 'model__min_samples_split': 2,
 'model__n_estimators': 200}

In [21]:
# лучшая модель
best_rf_pipeline = rf_grid.best_estimator_

In [22]:
# предсказания
best_rf_y_train_pred = best_rf_pipeline.predict(X_train)
best_rf_y_test_pred = best_rf_pipeline.predict(X_test)

In [23]:
# метрики
best_rf_train_metrics = metrics_helper.regression_metrics(y_train, best_rf_y_train_pred)
best_rf_test_metrics = metrics_helper.regression_metrics(y_test, best_rf_y_test_pred)

print('Метрики на train:')
display(best_rf_train_metrics)

print('Метрики на test:')
display(best_rf_test_metrics)

Метрики на train:


{'MAE': 70.93393588876484,
 'MSE': 19819.100585661654,
 'RMSE': np.float64(140.78032740998174),
 'R2': 0.8479864862719926}

Метрики на test:


{'MAE': 176.88550757121703,
 'MSE': 129786.51890590547,
 'RMSE': np.float64(360.25896089605527),
 'R2': 0.5301415758366048}

Подбор гиперпараметров немного улучшил качество модели. Метрики на тестовой выборке стали лучше, а разница между train и test уменьшилась, что говорит о снижении переобучения.

Улучшение незначительное, но в целом RandomForest остаётся хорошим решением для этой задачи.

#### Градиентный бустинг

In [24]:
gb_pipeline = Pipeline([
    (
        'preprocessor',
        DatasetPreprocessor(
            target_column='IC50, mM',
            drop_unnamed=True,
            drop_leakage=True,
            drop_constant_features=True,
            drop_corr=True,
            corr_threshold=0.9,
            log_features=True,
            log_skew_threshold=2.0,
            log_min_unique_values=10
        )
    ),
    ('scaler', StandardScaler()),
    (
        'model',
        GradientBoostingRegressor(
            n_estimators=100,
            learning_rate=0.1,
            max_depth=3,
            random_state=42
        )
    )
])

In [25]:
# обучение
gb_pipeline.fit(X_train, y_train)
# предсказания
gb_y_train_pred = gb_pipeline.predict(X_train)
gb_y_test_pred = gb_pipeline.predict(X_test)

In [26]:
# метрики
gb_train_metrics = metrics_helper.regression_metrics(y_train, gb_y_train_pred)
gb_test_metrics = metrics_helper.regression_metrics(y_test, gb_y_test_pred)

print('Метрики на train:')
display(gb_train_metrics)

print('Метрики на test:')
display(gb_test_metrics)

Метрики на train:


{'MAE': 90.16543383069926,
 'MSE': 24122.57551689482,
 'RMSE': np.float64(155.31444078673053),
 'R2': 0.8149786137547894}

Метрики на test:


{'MAE': 184.7145112394991,
 'MSE': 139159.27004902647,
 'RMSE': np.float64(373.0405742664281),
 'R2': 0.4962099616804745}

- Модель показывает хорошее качество на train ($R^2 \approx 0.81$), но хуже на test ($R^2 \approx 0.49$)
- Наблюдается переобучение (просадка между train и test)
- По качеству уступает RandomForest ($\approx 0.53$ на test)

**Необходимо выполнить подбор гиперпараметров**

In [27]:
# сетка гиперпараметров для GradientBoosting
gb_param_grid = {
    'model__n_estimators': [100, 200, 300],   # количество деревьев
    'model__learning_rate': [0.03, 0.05, 0.1],  # скорость обучения
    'model__max_depth': [2, 3, 4],  # глубина базовых деревьев
    'model__min_samples_split': [2, 5, 10],  # минимальное число объектов для разбиения узла
    'model__min_samples_leaf': [1, 2, 4]  # минимальное число объектов в листе
}

# подбор лучших параметров по кросс-валидации
gb_grid_search = GridSearchCV(
    estimator=gb_pipeline,
    param_grid=gb_param_grid,
    scoring='r2',
    cv=5,
    n_jobs=-1,
    verbose=1
)

In [28]:
gb_grid_search.fit(X_train, y_train)

Fitting 5 folds for each of 243 candidates, totalling 1215 fits


,"estimator estimator: estimator objectThis is assumed to implement the scikit-learn estimator interface.Either estimator needs to provide a ``score`` function,or ``scoring`` must be passed.",Pipeline(step...m_state=42))])
,"param_grid param_grid: dict or list of dictionariesDictionary with parameters names (`str`) as keys and lists ofparameter settings to try as values, or a list of suchdictionaries, in which case the grids spanned by each dictionaryin the list are explored. This enables searching over any sequenceof parameter settings.","{'model__learning_rate': [0.03, 0.05, ...], 'model__max_depth': [2, 3, ...], 'model__min_samples_leaf': [1, 2, ...], 'model__min_samples_split': [2, 5, ...], ...}"
,"scoring scoring: str, callable, list, tuple or dict, default=NoneStrategy to evaluate the performance of the cross-validated model onthe test set.If `scoring` represents a single score, one can use:- a single string (see :ref:`scoring_string_names`);- a callable (see :ref:`scoring_callable`) that returns a single value;- `None`, the `estimator`'s :ref:`default evaluation criterion ` is used.If `scoring` represents multiple scores, one can use:- a list or tuple of unique strings;- a callable returning a dictionary where the keys are the metric names and the values are the metric scores;- a dictionary with metric names as keys and callables as values.See :ref:`multimetric_grid_search` for an example.",'r2'
,"n_jobs n_jobs: int, default=NoneNumber of jobs to run in parallel.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary `for more details... versionchanged:: v0.20 `n_jobs` default changed from 1 to None",-1
,"refit refit: bool, str, or callable, default=TrueRefit an estimator using the best found parameters on the wholedataset.For multiple metric evaluation, this needs to be a `str` denoting thescorer that would be used to find the best parameters for refittingthe estimator at the end.Where there are considerations other than maximum score inchoosing a best estimator, ``refit`` can be set to a function whichreturns the selected ``best_index_`` given ``cv_results_``. In thatcase, the ``best_estimator_`` and ``best_params_`` will be setaccording to the returned ``best_index_`` while the ``best_score_``attribute will not be available.The refitted estimator is made available at the ``best_estimator_``attribute and permits using ``predict`` directly on this``GridSearchCV`` instance.Also for multiple metric evaluation, the attributes ``best_index_``,``best_score_`` and ``best_params_`` will only be available if``refit`` is set and all of them will be determined w.r.t this specificscorer.See ``scoring`` parameter to know more about multiple metricevaluation.See :ref:`sphx_glr_auto_examples_model_selection_plot_grid_search_digits.py`to see how to design a custom selection strategy using a callablevia `refit`.See :ref:`this example`for an example of how to use ``refit=callable`` to balance modelcomplexity and cross-validated score... versionchanged:: 0.20 Support for callable added.",True
,"cv cv: int, cross-validation generator or an iterable, default=NoneDetermines the cross-validation splitting strategy.Possible inputs for cv are:- None, to use the default 5-fold cross validation,- integer, to specify the number of folds in a `(Stratified)KFold`,- :term:`CV splitter`,- An iterable yielding (train, test) splits as arrays of indices.For integer/None inputs, if the estimator is a classifier and ``y`` iseither binary or multiclass, :class:`StratifiedKFold` is used. In allother cases, :class:`KFold` is used. These splitters are instantiatedwith `shuffle=False` so the splits will be the same across calls.Refer :ref:`User Guide ` for the variouscross-validation strategies that can be used here... versionchanged:: 0.22 ``cv`` default value if None changed from 3-fold to 5-fold.",5
,"verbose verbose: intControls the verbosity: the higher, the more messages.- >1 : the compu

In [29]:
# лучшие параметры
print('Лучшие параметры:')
display(gb_grid_search.best_params_)

# лучшая модель
best_gb_pipeline = gb_grid_search.best_estimator_

# предсказания
best_gb_y_train_pred = best_gb_pipeline.predict(X_train)
best_gb_y_test_pred = best_gb_pipeline.predict(X_test)

Лучшие параметры:


{'model__learning_rate': 0.03,
 'model__max_depth': 2,
 'model__min_samples_leaf': 2,
 'model__min_samples_split': 5,
 'model__n_estimators': 300}

In [30]:
# метрики лучшей модели GradientBoosting
best_gb_train_metrics = metrics_helper.regression_metrics(y_train, best_gb_y_train_pred)
best_gb_test_metrics = metrics_helper.regression_metrics(y_test, best_gb_y_test_pred)

print('Метрики на train:')
display(best_gb_train_metrics)

print('Метрики на test:')
display(best_gb_test_metrics)

Метрики на train:


{'MAE': 117.13781566318909,
 'MSE': 38388.25722362102,
 'RMSE': np.float64(195.9292148292873),
 'R2': 0.7055601064621988}

Метрики на test:


{'MAE': 200.02907063327075,
 'MSE': 140971.5734612112,
 'RMSE': np.float64(375.46181358589746),
 'R2': 0.4896489873009058}

- Подбор гиперпараметров снизил переобучение (train $R^2 \approx 0.71$, test $R^2 \approx 0.49$)
- Разрыв между train и test уменьшился
- Однако качество на test практически не улучшилось по сравнению с базовой версией
- По метрике $R^2$ модель уступает RandomForest ($\approx 0.53$ на test)

**Вывод:** RandomForest остаётся лучшей моделью для задачи регрессии IC50